# RAG Dataset Preparation

In this notebook, I prepare the dataset for my RAG-based question answering system.

I start from the full lecture text file, then parse it into page-level data and further split it into smaller chunks.

These chunks will be used later for embedding and retrieval.

The outputs include:
- a page-level CSV
- a chunk-level CSV
- a readable Markdown file for checking the chunks

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%cd /content/drive/MyDrive

/content/drive/MyDrive


In [19]:
import re
import csv
from pathlib import Path

# input / output
input_path = Path("ds593_dataset_full.txt")

pages_csv = Path("ds593_dataset_pages.csv")
chunks_csv = Path("ds593_chunks.csv")
chunks_md = Path("ds593_chunks.md")

# read txt
text = input_path.read_text(encoding="utf-8")

## Create page-Level dataset

In this part, I take the raw text file and split it into page-level data.

Each row includes:
- lecture name
- source file
- page number
- text content

Then I save everything into a CSV file.

In [20]:
# Parse full txt into pages
# find source file blocks
lecture_blocks = re.split(r"==============================", text)

page_rows = []

current_lecture = None
current_source = None

for block in lecture_blocks:
    block = block.strip()
    if not block:
        continue

    lecture_match = re.search(r"(Lecture \d+ - .+)", block)
    source_match = re.search(r"Source file:\s*(.+)", block)

    if lecture_match:
        current_lecture = lecture_match.group(1).strip()

    if source_match:
        current_source = source_match.group(1).strip()

    page_pattern = r"--- (Lecture \d+ - .+?) \| Page (\d+) ---"
    parts = re.split(page_pattern, block)

    for i in range(1, len(parts), 3):
        lecture_title = parts[i].strip()
        page_num = int(parts[i + 1])
        page_text = parts[i + 2].strip()

        page_rows.append({"lecture": lecture_title,
                          "source_file": current_source,
                          "page": page_num,
                          "text": page_text})


# save page-level CSV
with pages_csv.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["lecture", "source_file", "page", "text"])
    writer.writeheader()
    writer.writerows(page_rows)

## Create small chunks dataset

In this part, I split the page-level text into smaller chunks.

Chunking helps improve retrieval performance because shorter pieces of text are easier to match with a query. I also use overlap so that important context is not lost between chunks.

Each chunk includes:
- a unique chunk ID
- lecture information
- page number
- chunk index
- text content

Finally, I save all the chunks into a CSV file, which will be used later for embedding and retrieval in the RAG system.

In [37]:
# Create chunks CSV
def chunk_text(text, max_chars=400, overlap_paragraphs=1):
    paragraphs = [p.strip() for p in text.split("\n") if p.strip()]
    chunks = []
    current = []

    for p in paragraphs:
        candidate = "\n".join(current + [p])

        if len(candidate) <= max_chars:
            current.append(p)
        else:
            if current:
                chunks.append("\n".join(current))

            # keep last paragraph as overlap
            current = current[-overlap_paragraphs:] if overlap_paragraphs > 0 else []
            current.append(p)

    if current:
        chunks.append("\n".join(current))

    return chunks


chunk_rows = []

for row in page_rows:
    lecture = row["lecture"]
    source_file = row["source_file"]
    page = row["page"]
    page_text = row["text"]

    chunks = chunk_text(page_text, max_chars=400, overlap_paragraphs=1)

    lecture_num_match = re.search(r"Lecture (\d+)", lecture)
    lecture_num = lecture_num_match.group(1) if lecture_num_match else "X"

    for idx, chunk in enumerate(chunks, start=1):
        chunk_id = f"L{lecture_num}_p{page}_c{idx}"

        chunk_rows.append({"chunk_id": chunk_id,
                           "lecture": lecture,
                           "source_file": source_file,
                           "page": page,
                           "chunk_index": idx,
                           "text": chunk})


# save chunks CSV
with chunks_csv.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f,
                            fieldnames=["chunk_id",
                                        "lecture",
                                        "source_file",
                                        "page",
                                        "chunk_index",
                                        "text"])
    writer.writeheader()
    writer.writerows(chunk_rows)

## Create chunks markdown dataset

In this part, I create a Markdown version of the chunked dataset.

This makes it easier to look through the data, check if the chunks make sense, and better understand how everything is organized.

In [38]:
# Create Markdown
with chunks_md.open("w", encoding="utf-8") as f:
    f.write("# DS593 RAG Dataset Chunks\n\n")
    f.write("This file contains readable chunks generated from the full dataset text.\n\n")

    current_section = None

    for row in chunk_rows:
        section = f"{row['lecture']} | Page {row['page']}"

        if section != current_section:
            f.write(f"\n## {section}\n\n")
            current_section = section

        f.write(f"### {row['chunk_id']}\n\n")
        f.write(row["text"])
        f.write("\n\n---\n\n")

## Save Dataset Files

In this part, I save the processed data into CSV files.

This includes both the page-level data and the chunk-level data, so they can be used later for embedding and retrieval.

In [39]:
print("Done!")
print(f"Saved: {pages_csv}")
print(f"Saved: {chunks_csv}")
print(f"Saved: {chunks_md}")

Done!
Saved: ds593_dataset_pages.csv
Saved: ds593_chunks.csv
Saved: ds593_chunks.md
